# br_bonds_v2 — Portfolio Layer Examples

Examples for `Instrument`, `Position`, `Portfolio` classes.
Covers buy-and-hold TRI for a live bond and a matured bond,
rolling constant-maturity TRI, and comparison vs CDI.

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, r'C:\Users\Ivanildo\py_finance\fixed income')

from bizdays import Calendar
from br_bonds_v2.portfolio import Instrument, Position, Portfolio
from br_bonds_v2.analytics import risk_metrics
from br_bonds_v2.secondary import get_pu_series, parse_ativo

cal  = Calendar.load('ANBIMA')
DATA = r'C:\Users\Ivanildo\py_finance\fixed income\dados_bcb_secundario'

## 1. Load data

In [ ]:
# Secondary market prices (BCB)
raw = pd.read_parquet(DATA + r'\bcb_mercado_secundario_titulos_publicos_filtrado.parquet')
raw['data_mov'] = pd.to_datetime(raw['data_mov'])

# VNA (NTN-B IPCA accrual)
vna_s = pd.read_parquet(DATA + r'\vna_ntnb.parquet')['vna']
vna_s.index = pd.to_datetime(vna_s.index)

# CDI cumulative index
cdi_raw = pd.read_parquet(DATA + r'\cdi_series.parquet')
print(cdi_raw.dtypes)
print(cdi_raw.head())

In [ ]:
# Check available NTN-B ISINs by volume
ntnb = raw[raw['sigla'] == 'NTN-B']
ntnb.groupby(['vencimento', 'codigo_isin'])['quant_negociada'].sum().sort_values(ascending=False).head(10)

## 2. Instrument — per-date analytics

In [ ]:
# Describe a bond
ntnb35 = Instrument('NTNB', maturity='2035-05-15')
print(ntnb35)

# Reference date and du
date = pd.Timestamp('2026-03-12')
du   = ntnb35.du(date, cal)
ytm  = 0.0752
vna  = float(vna_s.loc[date])

print(f'\ndu={du}  ytm={ytm*100:.2f}%  VNA={vna:.4f}')

# Price — exact schedule (requires date + cal)
pu_exact   = ntnb35.price(ytm, du, vna=vna, date=date, cal=cal)
pu_uniform = ntnb35.price(ytm, du, vna=vna)  # fallback: uniform 126-bday spacing
print(f'\nPU exact   = {pu_exact:.6f}')
print(f'PU uniform = {pu_uniform:.6f}  (diff: {pu_exact - pu_uniform:.4f})')

# Full analytics row
row = ntnb35.analytics_row(ytm, du, vna=vna, date=date, cal=cal)
for k, v in row.items():
    print(f'  {k:12s} = {v:.6f}')

## 3. Buy-and-hold TRI — LIVE bond (NTN-B 2035-05-15)

In [ ]:
# Load price series for NTN-B 2035-05-15 (dominant ISIN by volume)
pu_live = get_pu_series(raw, 'NTN-B', '2035-05-15', 'BRSTNCNTB0O7')
print(f'Rows: {len(pu_live)}  |  {pu_live.index[0].date()} to {pu_live.index[-1].date()}')
print(pu_live.head())

In [ ]:
# Coupon dates in the window
cpn_dates = ntnb35.coupon_dates(pu_live.index[0], pu_live.index[-1], cal)
print('Coupon dates in window:', [str(d.date()) for d in cpn_dates])

# Coupon amounts on each date
for d in cpn_dates:
    vna_d = float(vna_s.loc[d])
    cpn_d = ntnb35.coupon_payment(vna=vna_d)
    pu_ex = pu_live.loc[d]
    pu_bef = pu_live.iloc[pu_live.index.get_loc(d) - 1]
    print(f'  {d.date()}:  VNA={vna_d:.2f}  coupon=R${cpn_d:.4f}  '
          f'PU_ex={pu_ex:.4f}  PU_before={pu_bef:.4f}  '
          f'gross_ret={(pu_ex + cpn_d)/pu_bef - 1:.4%}')

In [ ]:
# Build TRI
pos_live = Position(ntnb35, mode='buy_and_hold', entry_date=pu_live.index[0])

tri_live = pos_live.tri_series(
    prices    = pu_live,
    ytms      = pd.Series(np.nan, index=pu_live.index),  # not used in buy_and_hold
    cal       = cal,
    vna_series= vna_s,
    reinvest  = True,
)

print(tri_live.head())
print(tri_live.tail())

In [ ]:
s = risk_metrics(tri_live)
total = tri_live.iloc[-1] / tri_live.iloc[0] - 1

print(f'NTN-B 2035-05-15  (buy-and-hold)')
print(f'  Total return : {total*100:.2f}%')
print(f'  Ann return   : {s["ann_return"]*100:.2f}%')
print(f'  Ann vol      : {s["ann_vol"]*100:.2f}%')
print(f'  Sharpe       : {s["sharpe"]:.2f}')
print(f'  Max drawdown : {s["max_drawdown"]*100:.2f}%')
print(f'  VaR 95       : {s["var_95"]*100:.3f}%/day')
print(f'  CVaR 95      : {s["cvar_95"]*100:.3f}%/day')

## 4. Buy-and-hold TRI — MATURED bond (NTN-B 2025-05-15)

In [ ]:
pu_mat = get_pu_series(raw, 'NTN-B', '2025-05-15', 'BRSTNCNTB633')

# Add synthetic maturity day: cotacao=100 -> PU = VNA
# The TRI formula adds the coupon separately on coupon dates,
# so the price on maturity day should be par (VNA), not par + coupon.
vna_mat = float(vna_s.loc['2025-05-15'])
pu_mat.loc[pd.Timestamp('2025-05-15')] = vna_mat
pu_mat = pu_mat.sort_index()

print(f'Rows: {len(pu_mat)}  |  {pu_mat.index[0].date()} to {pu_mat.index[-1].date()}')
print(pu_mat.tail())

In [ ]:
ntnb25   = Instrument('NTNB', maturity='2025-05-15')
pos25    = Position(ntnb25, mode='buy_and_hold', entry_date=pu_mat.index[0])
tri_mat  = pos25.tri_series(
    prices    = pu_mat,
    ytms      = pd.Series(np.nan, index=pu_mat.index),
    cal       = cal,
    vna_series= vna_s,
    reinvest  = True,
)

s2 = risk_metrics(tri_mat)
total2 = tri_mat.iloc[-1] / tri_mat.iloc[0] - 1

print(f'NTN-B 2025-05-15  (matured, buy-and-hold)')
print(f'  Total return : {total2*100:.2f}%')
print(f'  Ann return   : {s2["ann_return"]*100:.2f}%')
print(f'  Ann vol      : {s2["ann_vol"]*100:.2f}%')
print(f'  Sharpe       : {s2["sharpe"]:.2f}')
print(f'  Max drawdown : {s2["max_drawdown"]*100:.2f}%')

# Final maturity payment breakdown
cpn_mat = ntnb25.coupon_payment(vna=vna_mat)
print(f'\n  At maturity (2025-05-15):')
print(f'    VNA             = R$ {vna_mat:.4f}')
print(f'    Coupon paid     = R$ {cpn_mat:.4f}')
print(f'    Total received  = R$ {vna_mat + cpn_mat:.4f}')

## 5. Rolling constant-maturity TRI

In [ ]:
# For rolling mode we need a daily (ytm, price) series at a fixed du_target.
# The breakeven_panel_bfuts has flat-forward bootstrapped real zero rates
# at a fixed du grid — perfect for constant-maturity NTN-B rolling.
DU_TARGET = 1260  # ~5 years

bp = pd.read_parquet(DATA + r'\breakeven_panel_bfuts.parquet')
bp['date'] = pd.to_datetime(bp['date'])

# Extract 5Y real zero rate series
roll_ytm = (bp[bp['du'] == DU_TARGET]
            .set_index('date')['r_real']
            .sort_index()
            / 100.0)   # convert % -> decimal

print(f'Rolling ytm series: {len(roll_ytm)} rows'
      f'  |  {roll_ytm.index[0].date()} to {roll_ytm.index[-1].date()}')
print(roll_ytm.tail(5))

In [ ]:
# Compute rolling price from the ytm series using Instrument.price()
# (uniform spacing — no exact date needed for constant-maturity, maturity is always du_target)
ntnb_5y = Instrument('NTNB')  # no fixed maturity

roll_vna   = vna_s.reindex(roll_ytm.index)          # VNA aligned to dates
roll_price = pd.Series(
    [ntnb_5y.price(y, DU_TARGET, vna=v)
     for y, v in zip(roll_ytm.values, roll_vna.values)],
    index=roll_ytm.index,
    name='price',
)

print(f'Price range: {roll_price.min():.2f} to {roll_price.max():.2f}')
print(roll_price.tail(5))

In [ ]:
# Build rolling TRI
# Return = (P_t / P_{t-1}) * (1 + ytm_{t-1})^(1/252)
# The accrual term (1+ytm)^(1/252) captures the daily carry implicit in the yield.
pos_roll = Position(ntnb_5y, mode='rolling', du_target=DU_TARGET, label='NTNB 5Y rolling')

tri_roll = pos_roll.tri_series(
    prices = roll_price,
    ytms   = roll_ytm,
)

s_roll   = risk_metrics(tri_roll)
total_r  = tri_roll.iloc[-1] / tri_roll.iloc[0] - 1
print(f'NTN-B 5Y rolling (2003 to today):')
print(f'  Total return : {total_r*100:.2f}%')
print(f'  Ann return   : {s_roll["ann_return"]*100:.2f}%')
print(f'  Ann vol      : {s_roll["ann_vol"]*100:.2f}%')
print(f'  Sharpe       : {s_roll["sharpe"]:.2f}')
print(f'  Max drawdown : {s_roll["max_drawdown"]*100:.2f}%')

## 6. CDI benchmark

In [ ]:
# CDI series: columns = ['date', 'cdi_annual'], cdi_annual is % p.a. annualized
cdi_raw = pd.read_parquet(DATA + r'\cdi_series.parquet')
cdi_raw['date'] = pd.to_datetime(cdi_raw['date'])
cdi_raw = cdi_raw.set_index('date').sort_index()
print(cdi_raw.head())
print(cdi_raw.tail())

In [ ]:
# Build CDI TRI rebased to 100 at the start of the buy-and-hold window
# cdi_annual is % p.a. -> daily factor = (1 + r/100)^(1/252)
cdi_factor = (1 + cdi_raw['cdi_annual'] / 100) ** (1 / 252)

start = tri_live.index[0]
end   = tri_live.index[-1]

cdi_window = cdi_factor.loc[start:end]
cdi_cum    = cdi_window.cumprod()
cdi_tri    = 100 * cdi_cum / cdi_cum.iloc[0]

print(f'CDI TRI: {cdi_tri.iloc[0]:.2f} -> {cdi_tri.iloc[-1]:.2f}'
      f'  total={(cdi_tri.iloc[-1]/cdi_tri.iloc[0]-1)*100:.2f}%')

## 7. Plot: NTN-B 2035 vs NTN-B 2025 vs CDI

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), gridspec_kw={'height_ratios': [3, 1]})

ax = axes[0]

# Rebase rolling TRI to 100 at the B&H start date so scales match
roll_window = tri_roll.reindex(tri_live.index).ffill().dropna()
roll_rebased = roll_window / roll_window.iloc[0] * 100

tri_live.plot(ax=ax, label='NTN-B 2035-05-15 (B&H)', color='steelblue', lw=1.5)
roll_rebased.plot(ax=ax, label='NTN-B 5Y rolling (CM, rebased)', color='royalblue', lw=1.5, linestyle='--')
cdi_tri.reindex(tri_live.index).ffill().plot(ax=ax, label='CDI', color='gray', linestyle=':', lw=1.5)

# Mark coupon dates on buy-and-hold series
for d in cpn_dates:
    if d in tri_live.index:
        ax.axvline(d, color='steelblue', linestyle=':', alpha=0.4, linewidth=0.8)
        ax.annotate('cpn', xy=(d, tri_live.loc[d]), fontsize=7, color='steelblue',
                    xytext=(5, 5), textcoords='offset points')

ax.set_title('Total Return Index — NTN-B B&H vs rolling vs CDI (rebased to 100 at window start)')
ax.set_ylabel('TRI')
ax.legend()
ax.grid(True, alpha=0.3)

# Drawdown panel
dd_bh   = (tri_live    / tri_live.cummax()    - 1) * 100
dd_roll = (roll_rebased / roll_rebased.cummax() - 1) * 100

ax2 = axes[1]
dd_bh.plot(ax=ax2, color='steelblue', label='B&H drawdown', lw=1)
dd_roll.plot(ax=ax2, color='royalblue', label='Rolling drawdown', lw=1, linestyle='--')
ax2.fill_between(dd_bh.index, dd_bh, 0, alpha=0.15, color='steelblue')
ax2.set_ylabel('Drawdown (%)')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Position-level analytics on a given date

In [ ]:
# Analytics for a R$1M notional position on 2026-03-12
date_ref  = pd.Timestamp('2026-03-12')
ytm_ref   = 0.0752
du_ref    = ntnb35.du(date_ref, cal)
vna_ref   = float(vna_s.loc[date_ref])

pos_1m = Position(ntnb35, quantity=1_000_000.0, mode='buy_and_hold')
a = pos_1m.analytics(date=date_ref, ytm=ytm_ref, du=du_ref, vna=vna_ref, cal=cal)

for k, v in a.items():
    print(f'  {k:12s}: {v}')

## 9. NTN-F buy-and-hold TRI (2031-01-01)

In [ ]:
# NTN-F 2031-01-01 — most liquid NTN-F by volume in the data window
# No VNA needed: coupons are fixed R$ amounts (face * (1.10^0.5 - 1) ~ 48.81)
pu_ntnf = get_pu_series(raw, 'NTN-F', '2031-01-01', 'BRSTNCNTF204')
print(f'Rows: {len(pu_ntnf)}  |  {pu_ntnf.index[0].date()} to {pu_ntnf.index[-1].date()}')
print(pu_ntnf.head(3))

# Coupon dates in window — NTN-F pays on Jan 1 and Jul 1 (ANBIMA-adjusted)
ntnf31 = Instrument('NTNF', maturity='2031-01-01')
cpn_dates_ntnf = ntnf31.coupon_dates(pu_ntnf.index[0], pu_ntnf.index[-1], cal)
print('\nCoupon dates:', [str(d.date()) for d in cpn_dates_ntnf])

# Coupon amount is constant (no VNA): face * (1.10^0.5 - 1)
print(f'Coupon per bond: R$ {ntnf31.coupon_payment():.6f}')

In [ ]:
pos_ntnf = Position(ntnf31, mode='buy_and_hold', entry_date=pu_ntnf.index[0],
                    label='NTN-F 2031 B&H')

tri_ntnf = pos_ntnf.tri_series(
    prices     = pu_ntnf,
    ytms       = pd.Series(np.nan, index=pu_ntnf.index),  # not used in buy_and_hold
    cal        = cal,
    vna_series = None,   # NTN-F coupons don't depend on VNA
    reinvest   = True,
)

# Cross-check: inspect each coupon date
for d in cpn_dates_ntnf:
    if d in pu_ntnf.index:
        cpn   = ntnf31.coupon_payment()
        p_ex  = pu_ntnf.loc[d]
        p_bef = pu_ntnf.iloc[pu_ntnf.index.get_loc(d) - 1]
        print(f'  {d.date()}:  coupon=R${cpn:.4f}  PU_ex={p_ex:.4f}  '
              f'PU_before={p_bef:.4f}  gross_ret={(p_ex+cpn)/p_bef-1:.4%}')

s_ntnf = risk_metrics(tri_ntnf)
print(f'\nNTN-F 2031-01-01 (buy-and-hold)')
print(f'  Total return : {(tri_ntnf.iloc[-1]/tri_ntnf.iloc[0]-1)*100:.2f}%')
print(f'  Ann return   : {s_ntnf["ann_return"]*100:.2f}%')
print(f'  Ann vol      : {s_ntnf["ann_vol"]*100:.2f}%')
print(f'  Sharpe       : {s_ntnf["sharpe"]:.2f}')
print(f'  Max drawdown : {s_ntnf["max_drawdown"]*100:.2f}%')

## 10. DI1 and DAP rolling TRI
`breakeven_panel_futures` has the DI1 (r_nominal) and DAP (r_real) flat-forward rates
at a fixed du grid, bootstrapped daily from B3 settlement prices — ideal for rolling.

In [ ]:
# ── DI1 settlement prices from parquet (no MySQL needed) ─────────────────────
# di1_futs.parquet: columns DATA, ATIVO, VENCIMENTO, AJUSTE_ATUAL
# VENCIMENTO uses B3 single-letter month codes: J26=Apr2026, N27=Jul2027, etc.

df_di1 = pd.read_parquet(DATA + r'\di1_futs.parquet')
df_di1['DATA'] = pd.to_datetime(df_di1['DATA'])

print('DI1 shape:', df_di1.shape)
print(df_di1.dtypes)
print(df_di1.head(5))

# ── DAP B&H: still requires MySQL (no individual contract parquet exists) ─────
# Uncomment to load DAP settlement prices:
#
# import mysql.connector
# conn   = mysql.connector.connect(host='localhost', user='root',
#                                  password='foxpro77', database='db_01')
# df_dap = pd.read_sql("SELECT * FROM db_01.ajuste_pregao WHERE ATIVO LIKE 'DAP%'", conn)
# conn.close()
# print('DAP shape:', df_dap.shape)
# print(df_dap.head(5))

In [ ]:
import importlib
import br_bonds_v2.secondary as _sec
importlib.reload(_sec)

from br_bonds_v2 import DI1Curve, di1_price, dap_price
from br_bonds_v2.secondary import _futures_to_zero_curve

# parse_ativo (secondary.py) handles ATIVO column format 'DAPK25', 'DI1N27', etc.


# ── DI1 rolling via DI1Curve flat-forward interpolator ───────────────────────
# _futures_to_zero_curve (secondary.py) handles VENCIMENTO parsing and
# AJUSTE_ATUAL → zero rate conversion in one step.
DU_FUT = 1260  # 5Y

di1_ytm_s = {}
for date, grp in df_di1.groupby('DATA'):
    # zero curve: columns du, zero_rate (decimal)
    zc = _futures_to_zero_curve(grp, date, cal, expiry_day=1)
    if len(zc) < 2:
        continue

    overnight = float(cdi_raw['cdi_annual'].get(date, np.nan))
    if np.isnan(overnight):
        continue

    # Prepend overnight vertex and build DI1Curve
    du_v   = np.concatenate([[1],                  zc['du'].values])
    rate_v = np.concatenate([[overnight / 100.0],  zc['zero_rate'].values])
    crv    = DI1Curve(date, du_v, rate_v)

    ytm = crv.ytm(DU_FUT)
    if ytm is not None:
        di1_ytm_s[date] = ytm

di1_ytm = pd.Series(di1_ytm_s, name='di1_ytm').sort_index()
di1_px  = pd.Series([di1_price(y, DU_FUT) for y in di1_ytm.values], index=di1_ytm.index)

print(f'DI1 rolling  rows={len(di1_ytm)}'
      f'  {di1_ytm.index[0].date()} to {di1_ytm.index[-1].date()}'
      f'  ytm: {di1_ytm.min()*100:.2f}%–{di1_ytm.max()*100:.2f}%')

In [ ]:
# ── DAP rolling (fixed maturity, from breakeven_panel_futures) ───────────────
# breakeven_panel_futures has flat-forward bootstrapped real zero rates
# from DAP settlements at a fixed du grid — same methodology as DI1Curve.
bpf = pd.read_parquet(DATA + r'\breakeven_panel_futures.parquet')
bpf['date'] = pd.to_datetime(bpf['date'])

dap_ytm = (bpf[bpf['du'] == DU_FUT]
           .set_index('date')['r_real']
           .sort_index()
           / 100.0)
dap_px  = pd.Series([dap_price(y, DU_FUT) for y in dap_ytm.values], index=dap_ytm.index)

tri_di1 = Position(Instrument('DI1'), mode='rolling', du_target=DU_FUT,
                   label='DI1 5Y rolling').tri_series(di1_px, di1_ytm)
tri_dap_roll = Position(Instrument('DAP'), mode='rolling', du_target=DU_FUT,
                        label='DAP 5Y rolling').tri_series(dap_px, dap_ytm)

for label, tri in [('DI1 5Y rolling', tri_di1), ('DAP 5Y rolling', tri_dap_roll)]:
    s = risk_metrics(tri)
    print(f'{label}:  total={(tri.iloc[-1]/tri.iloc[0]-1)*100:.1f}%'
          f'  ann={s["ann_return"]*100:.1f}%  vol={s["ann_vol"]*100:.1f}%'
          f'  sharpe={s["sharpe"]:.2f}  maxDD={s["max_drawdown"]*100:.2f}%')

In [ ]:
# ── DAP buy-and-hold: load individual contract prices from MySQL ───────────────────────
# ajuste_pregao schema: ATIVO = instrument label ('DAP - Cupom de IPCA'),
#                       VENCIMENTO = B3 month code ('N07', 'F25', ...) <- the actual contract
import mysql.connector
from br_bonds_v2.secondary import _parse_vencimento

conn = mysql.connector.connect(host='localhost', user='root',
                               password='foxpro77', database='db_01')
df_dap_raw = pd.read_sql(
    "SELECT DATA, VENCIMENTO, AJUSTE_ATUAL FROM ajuste_pregao WHERE ATIVO LIKE 'DAP%'",
    conn)
conn.close()

df_dap_raw['DATA']         = pd.to_datetime(df_dap_raw['DATA'])
df_dap_raw['AJUSTE_ATUAL'] = pd.to_numeric(df_dap_raw['AJUSTE_ATUAL'], errors='coerce')
df_dap_raw['maturity_date'] = _parse_vencimento(df_dap_raw['VENCIMENTO'], expiry_day=15)

# Inspect available contracts
print(df_dap_raw.groupby('VENCIMENTO').agg(
    maturity   = ('maturity_date', 'first'),
    first_date = ('DATA', 'min'),
    last_date  = ('DATA', 'max'),
    n_days     = ('DATA', 'nunique'),
).sort_values('maturity').to_string())

tri_dap_bh   = None
DAP_CONTRACT = None

In [ ]:
# ── DAP B&H TRI (runs only when MySQL data is loaded) ──────────────────────
# Pick a contract from the inspection table above, e.g.:
DAP_CONTRACT = 'F26'  # VENCIMENTO code
dap_bh = df_dap_raw[df_dap_raw['VENCIMENTO'] == DAP_CONTRACT].copy()
dap_bh = dap_bh[['DATA', 'AJUSTE_ATUAL', 'maturity_date']].dropna()
px_dap_bh = dap_bh.set_index('DATA')['AJUSTE_ATUAL'].sort_index()
mat_date  = dap_bh['maturity_date'].iloc[0]

# Add synthetic settlement at par if contract expired within data window
if mat_date not in px_dap_bh.index:
    px_dap_bh.loc[mat_date] = 100_000.0
    px_dap_bh = px_dap_bh.sort_index()
    print(f'Added synthetic settlement at {mat_date.date()}: R$ 100 000')

dap_inst_bh = Instrument('DAP', maturity=mat_date)
pos_dap_bh  = Position(dap_inst_bh, mode='buy_and_hold',
                       entry_date=px_dap_bh.index[0],
                       label=f'DAP {DAP_CONTRACT} B&H')
tri_dap_bh  = pos_dap_bh.tri_series(
    prices=px_dap_bh, ytms=pd.Series(np.nan, index=px_dap_bh.index), cal=cal)

s_bh = risk_metrics(tri_dap_bh)
print(f'DAP {DAP_CONTRACT}:  ann={s_bh["ann_return"]*100:.2f}%'
      f'  vol={s_bh["ann_vol"]*100:.2f}%  sharpe={s_bh["sharpe"]:.2f}')

## 11. Comparison plot — all instruments (rebased to common window)

In [ ]:
def rebase(tri: pd.Series, window_idx: pd.DatetimeIndex) -> pd.Series:
    """Reindex a TRI to window_idx and rebase to 100 at the first valid date."""
    s = tri.reindex(window_idx).ffill().dropna()
    return s / s.iloc[0] * 100


# Common window = NTN-B B&H window (Jan 2025 – Mar 2026)
idx = tri_live.index

series = {
    'NTN-B 2035 (B&H)'  : tri_live,
    'NTN-F 2031 (B&H)'  : rebase(tri_ntnf,    idx),
    'NTN-B 5Y rolling'  : rebase(tri_roll,    idx),
    'DI1 5Y rolling'    : rebase(tri_di1,     idx),
    'DAP 5Y rolling'    : rebase(tri_dap_roll, idx),
    'CDI'               : cdi_tri,
}

# Include DAP B&H only when MySQL data was loaded
if tri_dap_bh is not None:
    label_bh = f'DAP {DAP_CONTRACT} (B&H)'
    series[label_bh] = rebase(tri_dap_bh, idx)

colors = {
    'NTN-B 2035 (B&H)'  : 'steelblue',
    'NTN-F 2031 (B&H)'  : 'darkorange',
    'NTN-B 5Y rolling'  : 'royalblue',
    'DI1 5Y rolling'    : 'seagreen',
    'DAP 5Y rolling'    : 'mediumorchid',
    'CDI'               : 'gray',
}
if tri_dap_bh is not None:
    colors[label_bh] = 'orchid'

styles = {k: '-' for k in series}
for k in ('NTN-B 5Y rolling', 'DI1 5Y rolling', 'DAP 5Y rolling'):
    styles[k] = '--'
styles['CDI'] = ':'

fig, axes = plt.subplots(2, 1, figsize=(13, 9), gridspec_kw={'height_ratios': [3, 1]})
ax = axes[0]

for label, tri in series.items():
    tri.plot(ax=ax, label=label,
             color=colors.get(label, 'black'),
             linestyle=styles.get(label, '-'), lw=1.4)

ax.set_title('Total Return Index — all instruments rebased to 100 at window start')
ax.set_ylabel('TRI')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Drawdown panel — exclude CDI
ax2 = axes[1]
for label, tri in series.items():
    if label == 'CDI':
        continue
    dd = (tri / tri.cummax() - 1) * 100
    dd.plot(ax=ax2, label=label,
            color=colors.get(label, 'black'),
            linestyle=styles.get(label, '-'), lw=1)

ax2.set_ylabel('Drawdown (%)')
ax2.legend(fontsize=7, ncol=3)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()